# Step 2 – Do terrain inputs help? (SAR vs SAR + slope + HAND)
Adds **slope** (Copernicus DEM GLO-30) and **HAND** (MERIT Hydro) to every Sen1Floods11 chip via Google Earth Engine, then trains two identical U-Nets:
- **A:** VV, VH (same as notebook 01)
- **B:** VV, VH, slope, HAND

Results are also split by terrain: **flat (<5°)** vs **sloped (≥5°)** pixels.

Runtime: GPU (T4). Terrain download ≈ 5–10 min (only the first time, cached in Drive). Training ≈ 50 min for both models.

👉 **Edit `PROJECT` in the first cell** to your Earth Engine Cloud project ID.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID

!pip -q install segmentation-models-pytorch rasterio
import os, glob, random, numpy as np, pandas as pd, rasterio, torch, torch.nn as nn, ee
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Paper2_flood/02_terrain'
CACHE = f'{OUT}/terrain_chips'
os.makedirs(CACHE, exist_ok=True)

ee.Authenticate()
ee.Initialize(project=PROJECT)
print('Earth Engine OK')

## 1. Download Sen1Floods11 hand-labelled data (same as notebook 01)

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
B = 'gs://sen1floods11/v1.1'
import subprocess
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'data/flood_events/HandLabeled/LabelHand',
              'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'{B}/{p}', f'{ROOT}/'], check=True)

def chip_ids(csv):
    return [f.replace('_S1Hand.tif', '') for f in pd.read_csv(csv, header=None)[0]]
S = f'{ROOT}/flood_handlabeled'
split = {k: chip_ids(f'{S}/flood_{k}_data.csv') for k in ['train', 'valid', 'test', 'bolivia']}
ALL = [c for v in split.values() for c in v]
print({k: len(v) for k, v in split.items()})

## 2. Fetch slope and HAND for each chip from Earth Engine
Each chip's exact 512 × 512 grid (CRS + transform) is requested, so terrain pixels line up with the SAR pixels. Requests run in parallel threads.

In [ ]:
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic().setDefaultProjection('EPSG:4326', None, 30)
hand = ee.Image('MERIT/Hydro/v1_0_1').select('hnd')
TERRAIN = ee.Image.cat([ee.Terrain.slope(dem).rename('slope'),
                        hand.rename('hand')]).unmask(0).toFloat()

def fetch(cid, tries=4):
    out = f'{CACHE}/{cid}.npy'
    if os.path.exists(out): return 'cached'
    with rasterio.open(f'{ROOT}/S1Hand/{cid}_S1Hand.tif') as s:
        t, crs, h, w = s.transform, s.crs.to_string(), s.height, s.width
    req = {'expression': TERRAIN, 'fileFormat': 'NUMPY_NDARRAY',
           'grid': {'dimensions': {'width': w, 'height': h}, 'crsCode': crs,
                    'affineTransform': {'scaleX': t.a, 'shearX': t.b, 'translateX': t.c,
                                        'shearY': t.d, 'scaleY': t.e, 'translateY': t.f}}}
    for i in range(tries):
        try:
            a = ee.data.computePixels(req)
            np.save(out, np.stack([a['slope'], a['hand']]).astype('float32'))
            return 'ok'
        except Exception as e:
            err = str(e)
    return 'FAILED: ' + err[:120]

with ThreadPoolExecutor(8) as pool:
    status = list(pool.map(fetch, ALL))
print(pd.Series([s if not s.startswith('FAILED') else 'FAILED' for s in status]).value_counts())
for c, s in zip(ALL, status):
    if s.startswith('FAILED'): print(c, s)

## 3. Quick check: does the terrain line up with the SAR?

In [ ]:
cid = split['test'][0]
with rasterio.open(f'{ROOT}/S1Hand/{cid}_S1Hand.tif') as s: vv = s.read(1)
t = np.load(f'{CACHE}/{cid}.npy')
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(np.clip(vv, -25, 0), cmap='gray'); ax[0].set_title('VV')
ax[1].imshow(t[0], cmap='magma'); ax[1].set_title('Slope (°)')
ax[2].imshow(np.log1p(t[1]), cmap='viridis'); ax[2].set_title('log(1 + HAND m)')
for a in ax: a.axis('off')

## 4. How much steep terrain is in Sen1Floods11?
Useful for the paper: if few labelled pixels are steep, global training data under-represents mountains.

In [ ]:
rows = []
for k, ids in split.items():
    sl, lab = [], []
    for c in ids:
        with rasterio.open(f'{ROOT}/LabelHand/{c}_LabelHand.tif') as l: y = l.read(1)
        s = np.load(f'{CACHE}/{c}.npy')[0]; m = y >= 0
        sl.append(s[m]); lab.append(y[m])
    sl, lab = np.concatenate(sl), np.concatenate(lab)
    rows.append({'split': k, 'labelled px (M)': len(sl) / 1e6,
                 '% px slope ≥5°': 100 * (sl >= 5).mean(),
                 '% px slope ≥15°': 100 * (sl >= 15).mean(),
                 '% water px on slope ≥5°': 100 * (sl[lab == 1] >= 5).mean()})
terrain_stats = pd.DataFrame(rows).round(2)
terrain_stats.to_csv(f'{OUT}/sen1floods11_terrain_stats.csv', index=False); terrain_stats

## 5. Dataset with optional terrain channels

In [ ]:
def load(cid):
    with rasterio.open(f'{ROOT}/S1Hand/{cid}_S1Hand.tif') as s: x = s.read().astype('float32')
    with rasterio.open(f'{ROOT}/LabelHand/{cid}_LabelHand.tif') as l: y = l.read(1).astype('int64')
    x = np.clip(np.nan_to_num(x, nan=-50), -50, 1)
    t = np.nan_to_num(np.load(f'{CACHE}/{cid}.npy'))
    slope = t[0].copy()
    t = np.stack([np.clip(t[0], 0, 60), np.log1p(np.clip(t[1], 0, 200))])
    y[y < 0] = 255
    return np.concatenate([x, t]), y, slope

data = {c: load(c) for c in ALL}
tr = np.stack([data[c][0] for c in split['train']])
MEAN, STD = tr.mean((0, 2, 3)), tr.std((0, 2, 3)) + 1e-6
print('channel means', MEAN.round(2))

class Chips(Dataset):
    def __init__(self, ids, ch, train=False, crop=256):
        self.ids, self.ch, self.train, self.crop = ids, ch, train, crop
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x, y, s = data[self.ids[i]]
        x = ((x - MEAN[:, None, None]) / STD[:, None, None])[self.ch]
        if self.train:
            r, c = np.random.randint(0, 512 - self.crop, 2)
            x, y, s = x[:, r:r+self.crop, c:c+self.crop], y[r:r+self.crop, c:c+self.crop], s[r:r+self.crop, c:c+self.crop]
            if random.random() < .5: x, y, s = x[:, :, ::-1], y[:, ::-1], s[:, ::-1]
            if random.random() < .5: x, y, s = x[:, ::-1], y[::-1], s[::-1]
        return torch.tensor(x.copy()), torch.tensor(y.copy()), torch.tensor(s.copy())

## 6. Train and evaluate (same settings for both models)

In [ ]:
def scores(tp, fp, fn, tn):
    iou = tp / (tp + fp + fn + 1e-9); p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
    return dict(IoU=iou, F1=2*p*r/(p+r+1e-9), precision=p, recall=r, OA=(tp+tn)/(tp+tn+fp+fn+1e-9))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); acc = {k: np.zeros(4) for k in ['all', 'flat <5°', 'sloped ≥5°']}
    for x, y, s in loader:
        p = model(x.to(DEV)).argmax(1).cpu(); valid = y != 255
        for k, m in [('all', valid), ('flat <5°', valid & (s < 5)), ('sloped ≥5°', valid & (s >= 5))]:
            pp, yy = p[m], y[m]
            acc[k] += [((pp == 1) & (yy == 1)).sum(), ((pp == 1) & (yy == 0)).sum(),
                       ((pp == 0) & (yy == 1)).sum(), ((pp == 0) & (yy == 0)).sum()]
    return {k: scores(*v) for k, v in acc.items()}

def run(name, ch, epochs=60, seed=0):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    dl = {k: DataLoader(Chips(v, ch, train=(k == 'train')), batch_size=16 if k == 'train' else 4,
                        shuffle=(k == 'train'), num_workers=2) for k, v in split.items()}
    model = smp.Unet('resnet34', encoder_weights=None, in_channels=len(ch), classes=2).to(DEV)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1., 3.], device=DEV), ignore_index=255)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, path = 0, f'{OUT}/unet_{name}.pt'
    for ep in range(epochs):
        model.train()
        for x, y, _ in dl['train']:
            opt.zero_grad(); loss_fn(model(x.to(DEV)), y.to(DEV)).backward(); opt.step()
        sched.step()
        v = evaluate(model, dl['valid'])['all']['IoU']
        if v > best: best = v; torch.save(model.state_dict(), path)
        if ep % 10 == 0 or ep == epochs - 1: print(f'[{name}] epoch {ep:2d}  val IoU {v:.3f}')
    model.load_state_dict(torch.load(path))
    rows = []
    for k in ['test', 'bolivia']:
        for terr, m in evaluate(model, dl[k]).items():
            rows.append({'model': name, 'split': k, 'terrain': terr, **m})
    return pd.DataFrame(rows)

res = pd.concat([run('A_sar', [0, 1]), run('B_sar_terrain', [0, 1, 2, 3])])
res = res.round(3); res.to_csv(f'{OUT}/results_sar_vs_terrain.csv', index=False)

## 7. Comparison (IoU)

In [ ]:
pivot = res.pivot_table(index=['split', 'terrain'], columns='model', values='IoU')
pivot['gain (B − A)'] = pivot['B_sar_terrain'] - pivot['A_sar']
pivot.round(3)

In [ ]:
res[res.terrain == 'all'][['model', 'split', 'IoU', 'F1', 'precision', 'recall']]

## What to look for
- **gain (B − A)** on *all*: overall benefit of terrain.
- **gain on *sloped ≥5°***: the paper's key question, i.e. whether terrain cuts false water on slopes. Expect few sloped pixels here, which itself shows the data gap.
- Everything is saved in Google Drive under `Paper2_flood/02_terrain/`.